In [1]:
# =================
# 1. Imports
# =================

import os
import torch
import torchaudio
import pandas as pd
import numpy as np

from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, Wav2Vec2Model
from sklearn.metrics import roc_curve
from sklearn.metrics import accuracy_score


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

/home/jesse/Projects/MyEnvs/finquest_2026/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda


In [2]:
# =================
# 2. Paths
# =================

BASE_DIR = "/home/jesse/Projects/MyProjects/2026_FinQuest/Finquest_Wav2Vec_Deepfake/LA"

TRAIN_AUDIO_DIR = os.path.join(BASE_DIR, "ASVspoof2019_LA_train")
DEV_AUDIO_DIR   = os.path.join(BASE_DIR, "ASVspoof2019_LA_dev")
EVAL_AUDIO_DIR  = os.path.join(BASE_DIR, "ASVspoof2019_LA_eval")

PROTOCOL_DIR = os.path.join(BASE_DIR, "ASVspoof2019_LA_cm_protocols")

TRAIN_PROTOCOL = os.path.join(
    PROTOCOL_DIR, 
    "ASVspoof2019.LA.cm.train.trn.txt"
)

DEV_PROTOCOL   = os.path.join(
    PROTOCOL_DIR, 
    "ASVspoof2019.LA.cm.dev.trl.txt"
)

EVAL_PROTOCOL = os.path.join(
    PROTOCOL_DIR,
    "ASVspoof2019.LA.cm.eval.trl.txt"
)

In [3]:
# =================
# 3. Parse Protocols
# =================

def parse_protocol(protocol_path, split="train"):
    data = []

    if split == "train":
        base_dir = TRAIN_AUDIO_DIR
    elif split == "dev":
        base_dir = DEV_AUDIO_DIR
    else:
        base_dir = EVAL_AUDIO_DIR

    with open(protocol_path, "r") as f:
        for line in f:
            parts = line.strip().split()

            filename = parts[1]
            label = 1 if parts[-1] == "bonafide" else 0

            path = os.path.join(
                base_dir,
                "flac",
                filename + ".flac"
            )

            if os.path.exists(path):
                data.append((path, label))
            else:
                print("Missing:", path)

    return data

train_list = parse_protocol(TRAIN_PROTOCOL, split="train")
dev_list   = parse_protocol(DEV_PROTOCOL, split="dev")

print("Train:", len(train_list), "Dev:", len(dev_list))
print(train_list[:3])

Train: 25380 Dev: 24844
[('/home/jesse/Projects/MyProjects/2026_FinQuest/Finquest_Wav2Vec_Deepfake/LA/ASVspoof2019_LA_train/flac/LA_T_1138215.flac', 1), ('/home/jesse/Projects/MyProjects/2026_FinQuest/Finquest_Wav2Vec_Deepfake/LA/ASVspoof2019_LA_train/flac/LA_T_1271820.flac', 1), ('/home/jesse/Projects/MyProjects/2026_FinQuest/Finquest_Wav2Vec_Deepfake/LA/ASVspoof2019_LA_train/flac/LA_T_1272637.flac', 1)]


In [4]:
# =================
# 4. Dataset
# =================

class ASVDataset(Dataset):
    def __init__(self, data_list, processor, max_length=32000):
        self.data = data_list
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        import soundfile as sf

        path, label = self.data[idx]

        waveform, sr = sf.read(path)

        # 转 tensor
        waveform = torch.tensor(waveform, dtype=torch.float32)

        # stereo → mono
        if len(waveform.shape) > 1:
            waveform = waveform.mean(dim=1)

        # padding / truncation
        if len(waveform) > self.max_length:
            waveform = waveform[:self.max_length]
        else:
            pad_len = self.max_length - len(waveform)
            waveform = torch.nn.functional.pad(waveform, (0, pad_len))

        inputs = self.processor(
            waveform,
            sampling_rate=16000,
            return_tensors="pt",
            padding=True
        )

        return {
            "input_values": inputs.input_values.squeeze(0),
            "label": torch.tensor(label)
        }

In [5]:
# =================
# 5. Model
# =================

processor = AutoProcessor.from_pretrained("facebook/wav2vec2-base")

model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")

# 冻结/解冻 wav2vec2（关键！）
for name, param in model.named_parameters():
    # default freeze
    param.requires_grad = True

# 解冻最后 8 层
# for i in range(-8, 0):
    # for param in model.encoder.layers[i].parameters():
        # param.requires_grad = True

# ===== Attentive Pooling =====
class AttentivePooling(torch.nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attention = torch.nn.Sequential(
            torch.nn.Linear(hidden_size, 128),
            torch.nn.Tanh(),
            torch.nn.Linear(128, 1)
        )

    def forward(self, x):
        attn_weights = torch.softmax(self.attention(x), dim=1)
        pooled = torch.sum(attn_weights * x, dim=1)
        return pooled

pooling = AttentivePooling(768).to(device)

# ===== 新 classifier（MLP）=====
classifier = torch.nn.Sequential(
    torch.nn.Linear(768, 256),
    torch.nn.ReLU(),
    torch.nn.Dropout(0.3),
    torch.nn.Linear(256, 2)
)

model.to(device)
classifier.to(device)
model.to(device)
classifier.to(device)

Loading weights: 100%|██████████| 211/211 [00:00<00:00, 83114.03it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sequential(
  (0): Linear(in_features=768, out_features=256, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.3, inplace=False)
  (3): Linear(in_features=256, out_features=2, bias=True)
)

In [6]:
# =================
# 6. DataLoader
# =================
print("Train size:", len(train_list))
print("Dev size:", len(dev_list))

Train size: 25380
Dev size: 24844


In [7]:
# =================
# 7. Training Setup
# =================

train_dataset = ASVDataset(train_list, processor)
dev_dataset   = ASVDataset(dev_list, processor)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
dev_loader   = DataLoader(dev_dataset, batch_size=2)

In [8]:
# =================
# 8. Training Loop
# =================

optimizer = torch.optim.AdamW([
    # Layer‑wise LR decay
    {"params": model.encoder.layers[:6].parameters(), "lr": 5e-6},
    {"params": model.encoder.layers[6:9].parameters(), "lr": 1e-5},
    {"params": model.encoder.layers[9:].parameters(), "lr": 2e-5},
    {"params": classifier.parameters(), "lr": 5e-5},
])

criterion = torch.nn.CrossEntropyLoss()

# total training steps
num_train_steps = len(train_loader) * 20  # 20 epochs upper bound
# scheduler with warmup
scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer,
    start_factor=0.1,
    end_factor=1.0,
    total_iters=1000
)

def train_one_epoch():
    model.train()
    classifier.train()

    total_loss = 0

    for batch in tqdm(train_loader):
        optimizer.zero_grad()

        input_values = batch["input_values"].to(device)
        labels = batch["label"].to(device)

        outputs = model(input_values).last_hidden_state

        pooled = pooling(outputs)
        logits = classifier(pooled)

        loss = criterion(logits, labels)
        loss.backward()

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)

In [9]:
# =================
# 9. Evaluation
# =================

@torch.no_grad()
def evaluate_metrics(loader):
    model.eval()
    classifier.eval()

    preds = []
    labels_all = []
    scores = []

    for batch in loader:
        input_values = batch["input_values"].to(device)
        labels = batch["label"].cpu().numpy()

        outputs = model(input_values).last_hidden_state
        pooled = pooling(outputs)
        logits = classifier(pooled)

        prob = torch.softmax(logits, dim=1)[:,1].cpu().numpy()

        scores.append(prob)
        labels_all.extend(labels)

    scores = np.concatenate(scores)

    # accuracy at threshold 0.5
    preds = (scores >= 0.5).astype(int)
    acc = (preds == np.array(labels_all)).mean()

    # compute EER
    fpr, tpr, thresholds = roc_curve(labels_all, scores)
    fnr = 1 - tpr
    eer_threshold = thresholds[np.nanargmin(np.abs(fnr - fpr))]
    eer = fpr[np.nanargmin(np.abs(fnr - fpr))]

    return acc, eer, eer_threshold

In [10]:
# =====================
# 10. Main Training Loop with Early Stopping
# =====================

best_eer = 1.0
patience = 3
no_improve = 0
best_state = None

for epoch in range(20):
    loss = train_one_epoch()
    train_acc, train_eer, _ = evaluate_metrics(train_loader)
    dev_acc, dev_eer, _ = evaluate_metrics(dev_loader)

    print(f"Epoch {epoch} 🚀 loss={loss:.4f}, train_acc={train_acc:.4f}, train_EER={train_eer:.4f}")
    print(f"            dev_acc={dev_acc:.4f}, dev_EER={dev_eer:.4f}")

    # early stopping on EER (lower is better)
    if dev_eer < best_eer:
        best_eer = dev_eer
        no_improve = 0
        best_state = {
            "model": model.state_dict(),
            "classifier": classifier.state_dict(),
            "optimizer": optimizer.state_dict(),
        }
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

# restore best weights
if best_state is not None:
    model.load_state_dict(best_state["model"])
    classifier.load_state_dict(best_state["classifier"])

# File name with EER for easy reference
save_path = f"wav2vec2_deepfake_eer_{best_eer:.4f}.pth"

torch.save({
    "model": model.state_dict(),
    "classifier": classifier.state_dict()
}, save_path)

print(f"Model saved to {save_path}!")

100%|██████████| 12690/12690 [07:13<00:00, 29.30it/s]


Epoch 0 🚀 loss=0.0523, train_acc=0.9944, train_EER=0.0082
            dev_acc=0.9893, dev_EER=0.0179


100%|██████████| 12690/12690 [07:07<00:00, 29.69it/s]


Epoch 1 🚀 loss=0.0154, train_acc=0.9968, train_EER=0.0040
            dev_acc=0.9923, dev_EER=0.0127


100%|██████████| 12690/12690 [07:09<00:00, 29.51it/s]


Epoch 2 🚀 loss=0.0141, train_acc=0.9938, train_EER=0.0109
            dev_acc=0.9868, dev_EER=0.0216


... (6 outputs left out)

100%|██████████| 12690/12690 [06:58<00:00, 30.29it/s]


Epoch 6 🚀 loss=0.0235, train_acc=0.9966, train_EER=0.0117
            dev_acc=0.9868, dev_EER=0.0718


100%|██████████| 12690/12690 [06:57<00:00, 30.36it/s]


Epoch 7 🚀 loss=0.0163, train_acc=0.9992, train_EER=0.0014
            dev_acc=0.9920, dev_EER=0.0232
Early stopping at epoch 7
Model saved to wav2vec2_deepfake_eer_0.0105.pth!


In [11]:
# =====================
# 11. Evaluation on Eval Set
# =====================

eval_list = parse_protocol(EVAL_PROTOCOL, split="eval")

print("Eval size:", len(eval_list))
print(eval_list[:3])

eval_dataset = ASVDataset(eval_list, processor)

Eval size: 71237
[('/home/jesse/Projects/MyProjects/2026_FinQuest/Finquest_Wav2Vec_Deepfake/LA/ASVspoof2019_LA_eval/flac/LA_E_2834763.flac', 0), ('/home/jesse/Projects/MyProjects/2026_FinQuest/Finquest_Wav2Vec_Deepfake/LA/ASVspoof2019_LA_eval/flac/LA_E_8877452.flac', 0), ('/home/jesse/Projects/MyProjects/2026_FinQuest/Finquest_Wav2Vec_Deepfake/LA/ASVspoof2019_LA_eval/flac/LA_E_6828287.flac', 0)]


In [12]:
# =====================
# Evalulation
# =====================

# eval_list = eval_list[:2000]  # 先跑通 pipeline

eval_dataset = ASVDataset(eval_list, processor)
eval_loader  = DataLoader(eval_dataset, batch_size=2)

In [13]:
# ====================
# Evaulation
# ====================

from sklearn.metrics import accuracy_score

@torch.no_grad()
def evaluate_on_eval():
    model.eval()
    classifier.eval()

    preds = []
    labels_all = []

    for batch in tqdm(eval_loader):
        input_values = batch["input_values"].to(device)
        labels = batch["label"].to(device)

        outputs = model(input_values).last_hidden_state

        pooled = pooling(outputs)

        logits = classifier(pooled)

        pred = torch.argmax(logits, dim=1)

        preds.extend(pred.cpu().numpy())
        labels_all.extend(labels.cpu().numpy())

    acc = accuracy_score(labels_all, preds)

    print(f"Eval Accuracy: {acc:.4f}")

    return acc

evaluate_on_eval()

100%|██████████| 35619/35619 [04:40<00:00, 127.11it/s]


Eval Accuracy: 0.9697


0.9696786782149726

In [14]:
eval_data = DataLoader(eval_dataset, batch_size=8)

acc, eer, thresh = evaluate_metrics(eval_data)
print(f"Final Eval ⟹ Accuracy: {acc:.4f}, EER: {eer:.4f}, threshold={thresh:.4f}")

Final Eval ⟹ Accuracy: 0.9697, EER: 0.0393, threshold=0.1458
